# 핵심 아이디어 4: 다중 에이전트 태스크 스케줄링
- StrategyQA (YES/NO 추론) 데이터셋
- 5개 에이전트: ENC(질문 분해) → ALT(대안 확장) → VER(검증) → SYN(종합) → FIN(최종 답)
- 라우팅 조건: role_only / comm_only / role+comm
- 통신 지연 시뮬레이션 (거리 기반 채널 품질)

In [ ]:
!pip install -q torch transformers accelerate datasets bitsandbytes

import argparse
import csv
import json
import os
import random
import re
import statistics
import time
from collections import defaultdict
from dataclasses import dataclass, asdict
from typing import Any, Callable, Dict, List, Optional, Tuple

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


# =========================
# General utilities
# =========================


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def payload_bits_utf8(text: str) -> int:
    return len(text.encode("utf-8")) * 8


ANSWER_PATTERNS = [
    r"ANSWER\s*[:\-]?\s*(YES|NO)",
    r"FINAL\s+ANSWER\s*[:\-]?\s*(YES|NO)",
    r"CURRENT\s+BEST\s+CANDIDATE\s*[:\-]?\s*(YES|NO)",
    r"UPDATED\s+HYPOTHESIS\s*[:\-]?\s*(YES|NO)",
    r"REVISED\s+HYPOTHESIS\s*[:\-]?\s*(YES|NO)",
    r"INITIAL\s+HYPOTHESIS\s*[:\-]?\s*(YES|NO)",
]


def extract_answer(text: str) -> Optional[str]:
    """
    YES/NO parser.
    Prefer explicit answer lines near the end.
    """
    if not text or not text.strip():
        return None

    for line in reversed(text.strip().splitlines()):
        lu = line.upper()
        m = re.search(r"ANSWER\s*[:\-]?\s*(YES|NO)\b", lu)
        if m:
            return m.group(1)
        m = re.search(
            r"(?:FINAL\s+ANSWER|CURRENT\s+BEST\s+CANDIDATE|UPDATED\s+HYPOTHESIS|REVISED\s+HYPOTHESIS|INITIAL\s+HYPOTHESIS)\s*[:\-]?\s*(YES|NO)\b",
            lu,
        )
        if m:
            return m.group(1)

    upper = text.upper()
    for pattern in ANSWER_PATTERNS:
        matches = re.findall(pattern, upper)
        if matches:
            return matches[-1]

    tail = upper[-400:] if len(upper) > 400 else upper
    fallback = re.findall(r"\b(YES|NO)\b", tail)
    if fallback:
        return fallback[-1]
    return None


print("Cell 1 OK: imports + utilities loaded.")

In [ ]:
# =========================
# Link-quality model
# =========================

LINK_STATES = ["EXCELLENT", "GOOD", "FAIR", "POOR"]

LINK_STATE_WEIGHTS = {
    "EXCELLENT": 5,
    "GOOD": 20,
    "FAIR": 40,
    "POOR": 35,
}
_LINK_LABEL_WEIGHT_LIST = [LINK_STATE_WEIGHTS[s] for s in LINK_STATES]

_LINK_WEIGHTS_NEAR = (93, 4, 2, 1)
_LINK_WEIGHTS_FAR = (1, 2, 4, 93)

INTERMEDIATE_ROLES = ["SYN", "ALT", "VER"]
ALL_ROLES = ["ENC", "SYN", "ALT", "VER", "FIN"]


def undirected_pair_key(a: str, b: str) -> str:
    r0, r1 = sorted((a, b), key=lambda x: ALL_ROLES.index(x))
    return f"{r0}-{r1}"


_UNDIRECTED_ROLE_PAIRS: List[Tuple[str, str]] = [
    (ALL_ROLES[i], ALL_ROLES[j]) for i in range(5) for j in range(i + 1, 5)
]

ORCHESTRATOR_PREV_MESSAGE_MAX_CHARS = 3000

ORCHESTRATOR_ROUTING_PREAMBLE_ROLE_ONLY = """You are a routing decision agent.

Your goal:
Select the next agent based only on reasoning usefulness \u2014 use each agent's role description and the prior message below.

Final instruction:
After your reasoning (if any), the last line of your reply MUST be exactly:

Next: <SYN or ALT or VER>

Even if uncertain, you MUST choose one.
Do not omit the final line.

Write the role code only after \"Next:\" (one of SYN, ALT, VER). Example valid last line: Next: ALT
"""

ORCHESTRATOR_ROUTING_PREAMBLE_COMM = """You are a routing decision agent.

Your goal:
Use pairwise distances, channel label probabilities implied by distance (farther \u2192 worse labels on average), and the estimated hop times below to pick the next hop that minimizes communication delay for this step (and prefer lower projected time after that agent when comparing candidates).
There is no deadline or global time budget\u2014only compare the numbers given.
When you compute the comm_sec, you should estimate the communication delay for the next hop based on the link quality between the current agent and the next agent and the expected link quality based on the distance and according channel state PMF between the next agent and the remaining agents.

Final instruction:
After your reasoning (if any), the last line of your reply MUST be exactly:

Next: <SYN or ALT or VER>

Even if uncertain, you MUST choose one.
Do not omit the final line.

Write the role code only after \"Next:\" (one of SYN, ALT, VER). Example valid last line: Next: ALT
"""

ORCHESTRATOR_ROUTING_PREAMBLE_ROLE_COMM = """You are a routing decision agent.

Your goal:
Select the next agent using both (a) each agent's role and the reasoning task, and (b) communication: distance, channel label probabilities, comm_sec, and projected time after that agent.
When you compute the comm_sec, you should estimate the communication delay for the next hop based on the link quality between the current agent and the next agent and the expected link quality based on the distance and according channel state PMF between the next agent and the remaining agents.

Final instruction:
After your reasoning (if any), the last line of your reply MUST be exactly:

Next: <SYN or ALT or VER>

Even if uncertain, you MUST choose one.
Do not omit the final line.

Write the role code only after \"Next:\" (one of SYN, ALT, VER). Example valid last line: Next: ALT
"""


def _truncate_for_orchestrator(text: str, max_chars: int = ORCHESTRATOR_PREV_MESSAGE_MAX_CHARS) -> str:
    if not text or not text.strip():
        return "(no text yet)"
    t = text.strip()
    if len(t) <= max_chars:
        return t
    return t[: max_chars - 30].rstrip() + "\n... [truncated for orchestrator prompt]"


def _orchestrator_hint_lines_prefer_alt_after_enc(
    current_role: str,
    remaining: List[str],
    *,
    for_role_comm: bool,
) -> List[str]:
    if current_role != "ENC" or "ALT" not in remaining:
        return []
    out: List[str] = [
        "",
        "Pipeline hint (soft default for this experiment; not a strict rule):",
        (
            "After Agent ENC, Agent ALT is often a useful first intermediate step: "
            "it expands overlooked possibilities and missing reasoning before verification (VER) or synthesis (SYN). "
            "When SYN, ALT, and VER are all plausible, prefer ALT unless another role is clearly more useful for this message."
        ),
    ]
    if for_role_comm:
        out.append(
            "Role+Comm: if comm_sec / after_next_sec strongly favor SYN or VER over ALT for this hop, you may still pick the faster hop."
        )
    return out


def _orchestrator_hint_lines_prefer_ver_after_alt(
    current_role: str,
    remaining: List[str],
    *,
    for_role_comm: bool,
) -> List[str]:
    if current_role != "ALT" or "VER" not in remaining:
        return []
    out: List[str] = [
        "",
        "Pipeline hint (soft default for this experiment; not a strict rule):",
        (
            "After Agent ALT, Agent VER is often a useful next step: "
            "it stress-tests the expanded reasoning, checks weak assumptions, and corrects unsupported jumps before synthesis."
        ),
    ]
    if for_role_comm:
        out.append(
            "Role+Comm: if comm_sec / after_next_sec strongly favor another candidate, you may still pick the faster hop."
        )
    return out


def _orchestrator_hint_lines_prefer_syn_after_ver(
    current_role: str,
    remaining: List[str],
    *,
    for_role_comm: bool,
) -> List[str]:
    if current_role != "VER" or "SYN" not in remaining:
        return []
    out: List[str] = [
        "",
        "Pipeline hint (soft default for this experiment; not a strict rule):",
        (
            "After Agent VER, Agent SYN is often a useful next step: "
            "once the current reasoning has been stress-tested, SYN can integrate the verified state into the clearest pre-final verdict."
        ),
    ]
    if for_role_comm:
        out.append(
            "Role+Comm: if comm_sec / after_next_sec strongly favor another candidate, you may still pick the faster hop."
        )
    return out


@dataclass
class LinkQualityConfig:
    excellent_throughput_bps: float
    good_throughput_bps: float
    fair_throughput_bps: float
    poor_throughput_bps: float
    excellent_extra_delay_mean_ms: float
    good_extra_delay_mean_ms: float
    fair_extra_delay_mean_ms: float
    poor_extra_delay_mean_ms: float
    extra_delay_std_ms: float


@dataclass
class LinkStateParams:
    throughput_bps: float
    extra_delay_mean_sec: float
    extra_delay_std_sec: float


@dataclass
class LinkStateSample:
    state: str
    hidden_quality_score: float


@dataclass
class HopDelayBreakdown:
    state: str
    throughput_bps: float
    extra_delay_mean_sec: float
    extra_delay_std_sec: float
    random_extra_delay_sec: float
    serialization_delay_sec: float
    total_delay_sec: float
    hidden_quality_score: float


def build_link_state_table(cfg: LinkQualityConfig) -> Dict[str, LinkStateParams]:
    std_sec = cfg.extra_delay_std_ms / 1000.0
    return {
        "EXCELLENT": LinkStateParams(
            throughput_bps=cfg.excellent_throughput_bps,
            extra_delay_mean_sec=cfg.excellent_extra_delay_mean_ms / 1000.0,
            extra_delay_std_sec=std_sec,
        ),
        "GOOD": LinkStateParams(
            throughput_bps=cfg.good_throughput_bps,
            extra_delay_mean_sec=cfg.good_extra_delay_mean_ms / 1000.0,
            extra_delay_std_sec=std_sec,
        ),
        "FAIR": LinkStateParams(
            throughput_bps=cfg.fair_throughput_bps,
            extra_delay_mean_sec=cfg.fair_extra_delay_mean_ms / 1000.0,
            extra_delay_std_sec=std_sec,
        ),
        "POOR": LinkStateParams(
            throughput_bps=cfg.poor_throughput_bps,
            extra_delay_mean_sec=cfg.poor_extra_delay_mean_ms / 1000.0,
            extra_delay_std_sec=std_sec,
        ),
    }


def _representative_hidden_score_for_state(state: str) -> float:
    return {"EXCELLENT": 2.5, "GOOD": 1.7, "FAIR": 0.8, "POOR": 0.3}[state]


def _stable_edge_seed(example_seed: int, src: str, dst: str, salt: int) -> int:
    i, j = ALL_ROLES.index(src), ALL_ROLES.index(dst)
    return example_seed * 1_000_003 + i * 31 + j * 17 + salt


def _int_weights_blend_to_100(t: float) -> List[int]:
    t = max(0.0, min(1.0, t))
    raw = [
        _LINK_WEIGHTS_NEAR[k] * (1.0 - t) + _LINK_WEIGHTS_FAR[k] * t for k in range(4)
    ]
    flo = [int(x) for x in raw]
    err = 100 - sum(flo)
    frac_order = sorted(range(4), key=lambda k: raw[k] - flo[k], reverse=True)
    i = 0
    while err > 0:
        flo[frac_order[i % 4]] += 1
        err -= 1
        i += 1
    return flo


_PAIRWISE_DIST_MIN = 0.01
_PAIRWISE_DIST_MAX = 10_000.0


def sample_pairwise_distances(example_seed: int) -> Dict[Tuple[str, str], float]:
    rng = random.Random(example_seed * 1_009_901 + 246_913)
    lo, hi = _PAIRWISE_DIST_MIN, _PAIRWISE_DIST_MAX
    ratio = hi / lo
    n = len(_UNDIRECTED_ROLE_PAIRS)
    span = max(n - 1, 1)
    vals = [lo * (ratio ** (i / span)) for i in range(n)]
    rng.shuffle(vals)
    dist: Dict[Tuple[str, str], float] = {}
    for (a, b), v in zip(_UNDIRECTED_ROLE_PAIRS, vals):
        dist[(a, b)] = v
        dist[(b, a)] = v
    return dist


def precompute_edge_link_samples(
    example_seed: int,
) -> Tuple[Dict[Tuple[str, str], float], Dict[Tuple[str, str], LinkStateSample]]:
    dist_sym = sample_pairwise_distances(example_seed)
    d_vals = [dist_sym[(src, dst)] for src in ALL_ROLES for dst in ALL_ROLES if src != dst]
    d_min, d_max = min(d_vals), max(d_vals)
    span = d_max - d_min if d_max > d_min else 0.0

    out: Dict[Tuple[str, str], LinkStateSample] = {}
    for src in ALL_ROLES:
        for dst in ALL_ROLES:
            if src == dst:
                continue
            d = dist_sym[(src, dst)]
            if span > 0.0:
                t = (d - d_min) / span
            else:
                t = 0.5
            wlist = _int_weights_blend_to_100(t) if span > 0.0 else list(_LINK_LABEL_WEIGHT_LIST)
            rng = random.Random(_stable_edge_seed(example_seed, src, dst, salt=9401))
            state = rng.choices(LINK_STATES, weights=wlist, k=1)[0]
            score = _representative_hidden_score_for_state(state)
            out[(src, dst)] = LinkStateSample(state=state, hidden_quality_score=score)
    return dist_sym, out


def channel_label_pmf_for_edge(dist_sym: Dict[Tuple[str, str], float], src: str, dst: str) -> List[float]:
    d_vals = [dist_sym[(s, r)] for s in ALL_ROLES for r in ALL_ROLES if s != r]
    d_min, d_max = min(d_vals), max(d_vals)
    span = d_max - d_min if d_max > d_min else 0.0
    d = dist_sym[(src, dst)]
    if span > 0.0:
        t = (d - d_min) / span
        wlist = _int_weights_blend_to_100(t)
    else:
        wlist = list(_LINK_LABEL_WEIGHT_LIST)
    return [float(x) / 100.0 for x in wlist]


def format_channel_pmf_line(pmf: List[float]) -> str:
    return ", ".join(f"{LINK_STATES[i]}={pmf[i] * 100:.1f}%" for i in range(len(LINK_STATES)))


def hop_comm_random(example_seed: int, src: str, dst: str) -> random.Random:
    return random.Random(_stable_edge_seed(example_seed, src, dst, salt=12011))


def compute_comm_delay_from_state(
    payload_bits: int,
    state: str,
    table: Dict[str, LinkStateParams],
    hidden_quality_score: float,
    rng: random.Random,
) -> HopDelayBreakdown:
    params = table[state]
    serialization = payload_bits / params.throughput_bps if params.throughput_bps > 0 else float("inf")
    if params.extra_delay_std_sec > 0.0:
        raw = rng.gauss(params.extra_delay_mean_sec, params.extra_delay_std_sec)
    else:
        raw = params.extra_delay_mean_sec
    random_extra = max(0.0, raw)
    total = random_extra + serialization
    return HopDelayBreakdown(
        state=state,
        throughput_bps=params.throughput_bps,
        extra_delay_mean_sec=params.extra_delay_mean_sec,
        extra_delay_std_sec=params.extra_delay_std_sec,
        random_extra_delay_sec=random_extra,
        serialization_delay_sec=serialization,
        total_delay_sec=total,
        hidden_quality_score=hidden_quality_score,
    )


print("Cell 2 OK: link quality model loaded.")

In [ ]:
# =========================
# Role and prompt definitions
# =========================

ROLE_META = {
    "ENC": {
        "name": "Question Decomposer",
        "goal": (
            "You are the ONLY agent that sees the full question.\n\n"
            "Your job is NOT to solve it directly.\n"
            "Your job is to convert the question into a compact reasoning state for downstream agents.\n\n"
            "You MUST do all of the following:\n"
            "1) Rewrite the question as a clear claim to evaluate\n"
            "2) Break the problem into 2-4 atomic sub-questions\n"
            "3) List the key entities/concepts involved\n"
            "4) Give an initial hypothesis (YES or NO), but mark it as tentative\n\n"
            "Do NOT give a final answer.\n"
            "Do NOT write long explanations.\n"
            "Create a structured handoff that later agents can operate on."
        ),
        "format_tail": (
            "End with exactly this structure:\n"
            "Claim: <one sentence>\n"
            "Subquestions:\n"
            "- <subquestion 1>\n"
            "- <subquestion 2>\n"
            "- <subquestion 3 if needed>\n"
            "Key Concepts: <comma-separated>\n"
            "Initial Hypothesis: <YES/NO>"
        ),
    },

    "ALT": {
        "name": "Alternative Expander",
        "goal": (
            "You ONLY use the incoming message.\n\n"
            "Your job is to expand missing reasoning and overlooked possibilities.\n\n"
            "You MUST do all of the following:\n"
            "1) Add 2-4 factual assumptions, commonsense links, or missing considerations that could matter\n"
            "2) Explicitly consider why the opposite answer might still be plausible\n"
            "3) Update the hypothesis based on expanded possibilities\n\n"
            "Important constraints:\n"
            "- You MAY add plausible supporting facts or commonsense links\n"
            "- You MUST NOT finalize the answer\n"
            "- You MUST NOT only restate the input\n"
            "- You should widen the reasoning space before it gets narrowed again"
        ),
        "format_tail": (
            "End with exactly this structure:\n"
            "Added Considerations:\n"
            "- <item 1>\n"
            "- <item 2>\n"
            "- <item 3 if needed>\n"
            "Why YES might be true: <short>\n"
            "Why NO might be true: <short>\n"
            "Revised Hypothesis: <YES/NO>"
        ),
    },

    "VER": {
        "name": "Reasoning Verifier",
        "goal": (
            "You ONLY use the incoming message.\n\n"
            "Your job is to test the current reasoning for contradictions, weak assumptions, and unsupported jumps.\n\n"
            "You MUST do all of the following:\n"
            "1) Identify the weakest part of the current reasoning\n"
            "2) Point out any hidden assumption or logical gap\n"
            "3) State whether the current reasoning supports YES, supports NO, or remains mixed\n"
            "4) Update the hypothesis after verification\n\n"
            "Important constraints:\n"
            "- Do NOT introduce a completely new reasoning path\n"
            "- Do NOT simply confirm the input\n"
            "- Do NOT finalize the answer\n"
            "- Focus on checking, stress-testing, and correcting"
        ),
        "format_tail": (
            "End with exactly this structure:\n"
            "Main Weakness: <one sentence>\n"
            "Hidden Assumption: <one sentence>\n"
            "Verification Result: <supports YES / supports NO / mixed>\n"
            "Updated Hypothesis: <YES/NO>"
        ),
    },

    "SYN": {
        "name": "Decision Synthesizer",
        "goal": (
            "You ONLY use the incoming message.\n\n"
            "Your job is to synthesize the current reasoning state into the clearest pre-final verdict.\n\n"
            "You MUST do all of the following:\n"
            "1) Summarize the strongest reason for YES\n"
            "2) Summarize the strongest reason for NO\n"
            "3) Decide which side is better supported overall\n\n"
            "Important constraints:\n"
            "- Do NOT open new reasoning branches\n"
            "- Do NOT add many new facts\n"
            "- Resolve ambiguity using the reasoning already built\n"
            "- Prepare the clearest pre-final decision"
        ),
        "format_tail": (
            "End with exactly this structure:\n"
            "Best Reason for YES: <one sentence>\n"
            "Best Reason for NO: <one sentence>\n"
            "Current Best Candidate: <YES/NO>"
        ),
    },

    "FIN": {
        "name": "Final Answer Agent",
        "goal": (
            "You ONLY use the incoming message.\n\n"
            "Your job is to output the final answer.\n"
            "Do NOT add new reasoning.\n"
            "Do NOT reopen uncertainty.\n"
            "Be concise and decisive."
        ),
        "format_tail": "End with exactly one line:\nAnswer: <YES/NO>",
    },
}


def build_base_problem(example: Dict) -> str:
    return f"""Question:
{example['question']}

Task:
Decide whether the correct answer is YES or NO.

Allowed final labels:
YES
NO
"""


def normalize_strategyqa_row(row: Dict) -> Dict:
    raw_answer = row.get("answer", row.get("label", None))

    if isinstance(raw_answer, bool):
        answer = "YES" if raw_answer else "NO"
    else:
        s = str(raw_answer).strip().lower()
        if s in ("yes", "true", "1"):
            answer = "YES"
        elif s in ("no", "false", "0"):
            answer = "NO"
        else:
            raise ValueError(f"Cannot normalize StrategyQA answer: {raw_answer}")

    return {
        "task": "strategyqa",
        "question": str(row["question"]).strip(),
        "answer": answer,
    }


def normalize_task_arg(task: str) -> str:
    k = task.strip().lower().replace("-", "_")
    if k in ("strategyqa", "strategy_qa"):
        return "strategyqa"
    raise ValueError(
        f"Unknown --task {task!r}; currently this final code expects StrategyQA."
    )


def load_examples(
    dataset_name: str,
    split: str,
    strategyqa_dataset_id: str,
    strategyqa_dataset_config: Optional[str] = None,
):
    if dataset_name == "strategyqa":
        if not strategyqa_dataset_id:
            raise ValueError(
                "StrategyQA requires --strategyqa_dataset_id. "
                "Set it to the HF dataset repo name available in your environment."
            )
        if strategyqa_dataset_config:
            return load_dataset(strategyqa_dataset_id, strategyqa_dataset_config, split=split)
        return load_dataset(strategyqa_dataset_id, split=split)
    raise ValueError(f"Unknown dataset: {dataset_name}")


def prepare_example(raw: Dict, dataset_name: str) -> Dict:
    if dataset_name == "strategyqa":
        return normalize_strategyqa_row(raw)
    raise ValueError(f"Unknown dataset_name: {dataset_name}")


def build_agent_prompt(
    role: str,
    example: Dict,
    prev_output: Optional[str],
    prev_role: Optional[str],
) -> str:
    if role not in ROLE_META:
        raise ValueError(f"Unknown role: {role}")

    m = ROLE_META[role]
    body = build_base_problem(example) if role == "ENC" else (prev_output or "")

    return (
        f"Agent {role} ({m['name']}).\n\n"
        "Strict rules:\n"
        "- Follow ONLY your assigned operation.\n"
        "- Do NOT behave like another agent.\n"
        "- Keep the required output structure exactly.\n"
        "- Be concise, but do not omit required fields.\n\n"
        f"Your role:\n{m['goal']}\n\n"
        f"Input:\n{body}\n\n"
        f"{m['format_tail']}"
    )


print("Cell 3 OK: role and prompt definitions loaded.")

In [ ]:
# =========================
# Shared model wrapper
# =========================

def maybe_enable_attention_backend(attn_implementation: str) -> Optional[str]:
    if not torch.cuda.is_available():
        return None

    if hasattr(torch.backends.cuda, "enable_flash_sdp"):
        torch.backends.cuda.enable_flash_sdp(attn_implementation in {"auto", "sdpa", "flash_attention_2"})
    if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
        torch.backends.cuda.enable_mem_efficient_sdp(attn_implementation in {"auto", "sdpa", "flash_attention_2"})
    if hasattr(torch.backends.cuda, "enable_math_sdp"):
        torch.backends.cuda.enable_math_sdp(attn_implementation in {"auto", "eager"})

    if attn_implementation == "auto":
        return None
    return attn_implementation


class SharedLLM:
    def __init__(
        self,
        model_name: str,
        dtype: str = "float16",
        load_in_4bit: bool = False,
        quant_compute_dtype: str = "float16",
        device_map: str = "auto",
        torch_compile: bool = False,
        attn_implementation: str = "auto",
    ) -> None:
        torch_dtype = torch.bfloat16 if dtype == "bfloat16" else torch.float16
        bnb_compute_dtype = torch.bfloat16 if quant_compute_dtype == "bfloat16" else torch.float16

        quant_config = None
        if load_in_4bit:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=bnb_compute_dtype,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        model_kwargs = {
            "torch_dtype": torch_dtype,
            "device_map": device_map,
            "quantization_config": quant_config,
        }
        resolved_attn_impl = maybe_enable_attention_backend(attn_implementation)
        if resolved_attn_impl is not None:
            model_kwargs["attn_implementation"] = resolved_attn_impl

        self.model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        self.model.config.use_cache = True
        self.model.eval()

        if torch_compile and hasattr(torch, "compile"):
            try:
                self.model = torch.compile(self.model)  # type: ignore[assignment]
            except Exception as exc:
                print(f"[warn] torch.compile failed; continuing without compile: {exc}", flush=True)

    def _format_prompt(self, prompt: str, enable_thinking: Optional[bool] = None) -> str:
        if getattr(self.tokenizer, "chat_template", None):
            messages = [
                {"role": "system", "content": "You are a careful reasoning assistant."},
                {"role": "user", "content": prompt},
            ]
            base_kw: Dict[str, Any] = {"tokenize": False, "add_generation_prompt": True}
            if enable_thinking is None:
                return self.tokenizer.apply_chat_template(messages, **base_kw)
            try:
                return self.tokenizer.apply_chat_template(
                    messages, **base_kw, enable_thinking=enable_thinking
                )
            except TypeError:
                return self.tokenizer.apply_chat_template(messages, **base_kw)
        return prompt

    def generate(
        self,
        prompt: str,
        max_new_tokens: int,
        enable_thinking: Optional[bool] = None,
    ) -> Tuple[str, int, float]:
        formatted = self._format_prompt(prompt, enable_thinking=enable_thinking)
        inputs = self.tokenizer(formatted, return_tensors="pt", truncation=True, padding=False)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        prompt_len = int(inputs["input_ids"].shape[1])
        if max_new_tokens <= 0:
            mxl = getattr(self.tokenizer, "model_max_length", 32768) or 32768
            if mxl > 1_000_000:
                mxl = 32768
            max_new_tokens = max(1, mxl - prompt_len - 8)

        start = time.perf_counter()
        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        end = time.perf_counter()

        gen_ids = outputs[0][prompt_len:]
        text = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        return text, int(gen_ids.shape[0]), float(end - start)


print("Cell 4 OK: SharedLLM model wrapper loaded.")

In [ ]:
# =========================
# Delay estimation and orchestration
# =========================

@dataclass
class RouteDecision:
    chosen_next: str


class DelayEstimator:
    def __init__(
        self,
        roles: List[str],
        initial_compute_sec: float,
        max_new_tokens: int,
        default_message_bytes: int,
        link_state_table: Dict[str, LinkStateParams],
    ) -> None:
        self.compute_history: Dict[str, List[float]] = {r: [] for r in roles}
        self.token_history: Dict[str, List[int]] = {r: [] for r in roles}
        self.bit_history: Dict[str, List[int]] = {r: [] for r in roles}
        self.initial_compute_sec = initial_compute_sec
        self.max_new_tokens = max_new_tokens
        self.default_message_bytes = default_message_bytes
        self.link_state_table = link_state_table

    def observe(self, role: str, compute_sec: float, gen_tokens: int, message_bits: int) -> None:
        self.compute_history[role].append(compute_sec)
        self.token_history[role].append(gen_tokens)
        self.bit_history[role].append(message_bits)

    def estimate_compute(self, role: str) -> float:
        hist = self.compute_history[role]
        if hist:
            return statistics.mean(hist)
        return self.initial_compute_sec

    def estimate_output_bits(self, role: str) -> float:
        hist = self.bit_history[role]
        if hist:
            return statistics.mean(hist)
        return float(self.default_message_bytes * 8)

    def estimate_comm_from_state(self, bits: float, state: str) -> float:
        params = self.link_state_table[state]
        serialization = bits / params.throughput_bps if params.throughput_bps > 0 else float("inf")
        return params.extra_delay_mean_sec + serialization


def role_only_priority(visited: List[str], remaining: List[str]) -> List[str]:
    preferred = ["ALT", "VER", "SYN"]
    ordered = [r for r in preferred if r in remaining]
    if "FIN" in remaining and len(ordered) == 0:
        ordered.append("FIN")
    return ordered


def pick_min_immediate_comm_candidate(
    candidates: List[str],
    immediate_comm_sec: Dict[str, float],
    candidate_states: Dict[str, str],
) -> str:
    quality_rank = {"EXCELLENT": 0, "GOOD": 1, "FAIR": 2, "POOR": 3}
    role_tie = {"ALT": 0, "VER": 1, "SYN": 2}

    return min(
        candidates,
        key=lambda c: (
            immediate_comm_sec.get(c, float("inf")),
            quality_rank.get(candidate_states.get(c, "POOR"), 99),
            role_tie.get(c, 9),
        ),
    )


def parse_orchestrator_next(text: str, candidates: List[str]) -> Optional[str]:
    if not text or not candidates:
        return None
    block = text.strip()
    if "</think>" in block:
        block = block.rsplit("</think>", 1)[-1].strip()

    cand_pat = "|".join(re.escape(c) for c in sorted(candidates, key=len, reverse=True))
    last_explicit: Optional[str] = None
    for line in block.splitlines():
        m = re.search(
            rf"\b(?:NEXT|CHOICE|ROUTE|SELECT)\s*[:=]\s*({cand_pat})\b",
            line,
            re.I,
        )
        if m:
            tok = m.group(1).upper()
            if tok in candidates:
                last_explicit = tok

    if last_explicit is not None:
        return last_explicit

    nonempty = [ln.strip() for ln in block.splitlines() if ln.strip()]
    if not nonempty:
        return None
    last_line = nonempty[-1]
    mentioned = [c for c in candidates if re.search(rf"\b{re.escape(c)}\b", last_line, re.I)]
    if len(mentioned) == 1:
        return mentioned[0]
    return None


def build_orchestrator_prompt_role_only(
    visited: List[str],
    remaining: List[str],
    current_role: str,
    previous_agent_output: str,
) -> str:
    prev = _truncate_for_orchestrator(previous_agent_output)
    lines = [
        ORCHESTRATOR_ROUTING_PREAMBLE_ROLE_ONLY.rstrip(),
        "",
        "---",
        "Task (multi-agent YES/NO reasoning pipeline):",
        "Situation:",
        "  Visited (order): " + " -> ".join(visited) + f". Current node: {current_role}.",
        "  Choose exactly ONE next agent from: " + ", ".join(remaining) + ".",
        "",
        f"Message produced by agent {current_role}:",
        "---",
        prev,
        "---",
        "",
        "Remaining agents \u2014 what each can do:",
    ]
    for r in INTERMEDIATE_ROLES:
        if r in remaining:
            meta = ROLE_META[r]
            lines.append(f"- Agent {r} ({meta['name']}): {meta['goal']}")
    lines.extend(
        [
            "",
            "Decision guidance:",
            "",
            "Select the next agent based on overall usefulness of each role for this reasoning state.",
            "",
            "Compare all candidates before deciding.",
        ]
    )
    lines.extend(_orchestrator_hint_lines_prefer_alt_after_enc(current_role, remaining, for_role_comm=False))
    lines.extend(_orchestrator_hint_lines_prefer_ver_after_alt(current_role, remaining, for_role_comm=False))
    lines.extend(_orchestrator_hint_lines_prefer_syn_after_ver(current_role, remaining, for_role_comm=False))
    lines.extend(
        [
            "",
            "There is no mandatory fixed pipeline order; the soft hints above apply when competing roles are similarly plausible.",
            "",
            "Remember: your response must end with one line exactly of the form Next: SYN or Next: ALT or Next: VER.",
        ]
    )
    return "\n".join(lines)


def build_orchestrator_prompt_comm_only(
    visited: List[str],
    remaining: List[str],
    current_role: str,
    previous_agent_output: str,
    candidate_states: Dict[str, str],
    channel_pmfs: Dict[str, List[float]],
    immediate_comm_sec: Dict[str, float],
    projected_after_next_agent_sec: Dict[str, float],
    myopic_projection: bool,
    pairwise_distances: Dict[str, float],
) -> str:
    cands = [c for c in remaining if c in INTERMEDIATE_ROLES]
    lines = [
        ORCHESTRATOR_ROUTING_PREAMBLE_COMM.rstrip(),
        "",
        "---",
        "Task (multi-agent YES/NO reasoning pipeline):",
        "Situation:",
        "  Visited (order): " + " -> ".join(visited) + f". Current node: {current_role}.",
        "  Choose exactly ONE next agent from: " + ", ".join(remaining) + ".",
        "",
        "All pairwise distances this episode:",
    ]
    for k in sorted(pairwise_distances.keys()):
        lines.append(f"  {k}: {pairwise_distances[k]:.4f}")
    lines.extend(
        [
            "",
            f"From {current_role} to each candidate \u2014 distance, P(label|distance), sampled link for this hop, comm_sec, projected time after that agent:",
        ]
    )
    if not myopic_projection:
        lines.append("(after_next_sec includes a rough forecast of later hops.)")
    for cand in sorted(cands):
        pk = undirected_pair_key(current_role, cand)
        du = pairwise_distances.get(pk, float("nan"))
        pmf_s = format_channel_pmf_line(channel_pmfs[cand])
        st = candidate_states.get(cand, "?")
        ic = immediate_comm_sec.get(cand, float("nan"))
        pa = projected_after_next_agent_sec.get(cand, float("nan"))
        lines.append(
            f"  -> {cand}: dist_{pk}={du:.3f}; P(label|dist) [{pmf_s}]; sampled_link={st}; comm_sec={ic:.4f}; after_next_sec={pa:.4f}"
        )
    lines.extend(
        [
            "",
            "Decision guidance:",
            "- Use distance, sampled link, comm_sec, and after_next_sec to pick the fastest / most favorable hop.",
            "- Compare all candidates before deciding.",
            "",
            "Remember: your response must end with one line exactly of the form Next: SYN or Next: ALT or Next: VER.",
        ]
    )
    return "\n".join(lines)


def build_orchestrator_prompt_role_comm(
    visited: List[str],
    remaining: List[str],
    current_role: str,
    previous_agent_output: str,
    candidate_states: Dict[str, str],
    channel_pmfs: Dict[str, List[float]],
    immediate_comm_sec: Dict[str, float],
    projected_after_next_agent_sec: Dict[str, float],
    myopic_projection: bool,
    pairwise_distances: Dict[str, float],
) -> str:
    cands = [c for c in remaining if c in INTERMEDIATE_ROLES]
    prev = _truncate_for_orchestrator(previous_agent_output)
    lines = [
        ORCHESTRATOR_ROUTING_PREAMBLE_ROLE_COMM.rstrip(),
        "",
        "---",
        "Task (multi-agent YES/NO reasoning pipeline):",
        "Situation:",
        "  Visited (order): " + " -> ".join(visited) + f". Current node: {current_role}.",
        "  Choose exactly ONE next agent from: " + ", ".join(remaining) + ".",
        "",
        f"Message produced by agent {current_role}:",
        "---",
        prev,
        "---",
        "",
        "Agent roles (capabilities):",
    ]
    for r in INTERMEDIATE_ROLES:
        if r in remaining:
            meta = ROLE_META[r]
            lines.append(f"- Agent {r} ({meta['name']}): {meta['goal']}")

    lines.extend(["", "All pairwise distances this episode (smaller = closer):"])
    for k in sorted(pairwise_distances.keys()):
        lines.append(f"  {k}: {pairwise_distances[k]:.4f}")

    lines.append("")
    lines.append(
        f"From {current_role} to each candidate \u2014 distance, P(label|distance), sampled link for this hop, comm_sec, projected time after that agent:"
    )
    if not myopic_projection:
        lines.append("(after_next_sec includes a rough forecast of later hops.)")
    for cand in sorted(cands):
        pk = undirected_pair_key(current_role, cand)
        du = pairwise_distances.get(pk, float("nan"))
        pmf_s = format_channel_pmf_line(channel_pmfs[cand])
        st = candidate_states.get(cand, "?")
        ic = immediate_comm_sec.get(cand, float("nan"))
        pa = projected_after_next_agent_sec.get(cand, float("nan"))
        lines.append(
            f"  -> {cand}: dist_{pk}={du:.3f}; P(label|dist) [{pmf_s}]; sampled_link={st}; comm_sec={ic:.4f}; after_next_sec={pa:.4f}"
        )

    lines.extend(
        [
            "",
            "Decision guidance:",
            "- Use both reasoning usefulness and communication numbers.",
            "- If a role is only slightly better for reasoning but much slower in comm_sec / after_next_sec, you may choose the faster hop.",
            "- Reasoning contribution is slightly more important than communication cost unless timing differences are clearly large.",
            "- Compare all candidates before deciding.",
        ]
    )
    lines.extend(_orchestrator_hint_lines_prefer_alt_after_enc(current_role, remaining, for_role_comm=True))
    lines.extend(_orchestrator_hint_lines_prefer_ver_after_alt(current_role, remaining, for_role_comm=True))
    lines.extend(_orchestrator_hint_lines_prefer_syn_after_ver(current_role, remaining, for_role_comm=True))
    lines.extend(
        [
            "",
            "Remember: your response must end with one line exactly of the form Next: SYN or Next: ALT or Next: VER.",
        ]
    )
    return "\n".join(lines)


class Orchestrator:
    def __init__(
        self,
        policy_name: str,
        estimator: DelayEstimator,
        state_table: Dict[str, LinkStateParams],
        myopic_projection: bool = True,
    ) -> None:
        self.policy_name = policy_name
        self.estimator = estimator
        self.state_table = state_table
        self.myopic_projection = myopic_projection

    def _estimate_future_delay(self, chosen: str, remaining: List[str]) -> float:
        future = [r for r in remaining if r != chosen]
        if len([r for r in future if r in INTERMEDIATE_ROLES]) == 0 and "FIN" not in future:
            future.append("FIN")

        est = 0.0
        prev_role = chosen
        ordered = role_only_priority([chosen], [r for r in future if r != "FIN"]) + (["FIN"] if "FIN" in future else [])
        for r in ordered:
            est += self.estimator.estimate_compute(r)
            bits = self.estimator.estimate_output_bits(prev_role)
            est += self.estimator.estimate_comm_from_state(bits, "GOOD")
            prev_role = r
        return est

    def decide(
        self,
        current_role: str,
        current_message_bits: int,
        visited: List[str],
        remaining: List[str],
        candidate_states: Dict[str, str],
        elapsed_without_orchestrator: float,
        model: Optional["SharedLLM"] = None,
        orch_max_new_tokens: int = 256,
        previous_agent_output: str = "",
        dist_sym: Optional[Dict[Tuple[str, str], float]] = None,
        pairwise_distances: Optional[Dict[str, float]] = None,
        orchestrator_enable_thinking: bool = False,
    ) -> Tuple[RouteDecision, float, Optional[str]]:
        if len([r for r in remaining if r in INTERMEDIATE_ROLES]) == 0:
            return (
                RouteDecision(chosen_next="FIN"),
                0.0,
                None,
            )

        projected_delay_if_chosen: Dict[str, float] = {}
        immediate_comm_sec: Dict[str, float] = {}
        candidates = [r for r in remaining if r in INTERMEDIATE_ROLES]

        if self.policy_name == "role_only":
            fallback = role_only_priority(visited, remaining)[0]
            if model is None:
                raise ValueError("role_only policy requires a SharedLLM model")
            prompt = build_orchestrator_prompt_role_only(
                visited,
                remaining,
                current_role,
                previous_agent_output,
            )
            raw_text, _tok, orch_sec = model.generate(
                prompt, orch_max_new_tokens, enable_thinking=orchestrator_enable_thinking
            )
            parsed = parse_orchestrator_next(raw_text, candidates)
            chosen_next = parsed if parsed is not None else fallback
            return (
                RouteDecision(chosen_next=chosen_next),
                orch_sec,
                raw_text,
            )

        if self.policy_name not in ("link_quality_aware", "link_quality_llm"):
            raise ValueError(f"Unknown orchestrator policy: {self.policy_name}")

        if dist_sym is None or pairwise_distances is None:
            raise ValueError("link_quality_aware / link_quality_llm require dist_sym and pairwise_distances")

        channel_pmfs = {c: channel_label_pmf_for_edge(dist_sym, current_role, c) for c in candidates}

        for cand in candidates:
            immediate_comm = self.estimator.estimate_comm_from_state(float(current_message_bits), candidate_states[cand])
            compute_next = self.estimator.estimate_compute(cand)
            immediate_comm_sec[cand] = immediate_comm
            tail = 0.0 if self.myopic_projection else self._estimate_future_delay(cand, remaining)
            projected = elapsed_without_orchestrator + immediate_comm + compute_next + tail
            projected_delay_if_chosen[cand] = projected

        chosen_link_baseline = pick_min_immediate_comm_candidate(
            candidates,
            immediate_comm_sec,
            candidate_states,
        )

        if self.policy_name == "link_quality_aware":
            if model is None:
                raise ValueError("link_quality_aware policy requires a SharedLLM model")
            prompt = build_orchestrator_prompt_comm_only(
                visited=visited,
                remaining=remaining,
                current_role=current_role,
                previous_agent_output=previous_agent_output,
                candidate_states=candidate_states,
                channel_pmfs=channel_pmfs,
                immediate_comm_sec=immediate_comm_sec,
                projected_after_next_agent_sec=projected_delay_if_chosen,
                myopic_projection=self.myopic_projection,
                pairwise_distances=pairwise_distances,
            )
            raw_text, _tok, orch_sec = model.generate(
                prompt, orch_max_new_tokens, enable_thinking=orchestrator_enable_thinking
            )
            parsed = parse_orchestrator_next(raw_text, candidates)
            chosen_next = parsed if parsed is not None else chosen_link_baseline
            return (
                RouteDecision(chosen_next=chosen_next),
                orch_sec,
                raw_text,
            )

        if model is None:
            raise ValueError("link_quality_llm policy requires a SharedLLM model")

        prompt = build_orchestrator_prompt_role_comm(
            visited=visited,
            remaining=remaining,
            current_role=current_role,
            previous_agent_output=previous_agent_output,
            candidate_states=candidate_states,
            channel_pmfs=channel_pmfs,
            immediate_comm_sec=immediate_comm_sec,
            projected_after_next_agent_sec=projected_delay_if_chosen,
            myopic_projection=self.myopic_projection,
            pairwise_distances=pairwise_distances,
        )
        raw_text, _tok, orch_sec = model.generate(
            prompt, orch_max_new_tokens, enable_thinking=orchestrator_enable_thinking
        )
        parsed = parse_orchestrator_next(raw_text, candidates)
        chosen_next = parsed if parsed is not None else chosen_link_baseline

        return (
            RouteDecision(chosen_next=chosen_next),
            orch_sec,
            raw_text,
        )


print("Cell 5 OK: orchestrator and routing loaded.")

In [ ]:
# =========================
# Episode records and pipeline execution
# =========================

@dataclass
class HopRecord:
    src: str
    dst: str
    link_state: str
    hidden_quality_score: float
    throughput_bps: float
    extra_delay_mean_sec: float
    extra_delay_std_sec: float
    random_extra_delay_sec: float
    message_bits: int
    serialization_delay_sec: float
    communication_delay_sec: float


@dataclass
class AgentRecord:
    role: str
    role_name: str
    prompt_preview: str
    generated_text: str
    generated_tokens: int
    generated_utf8_bits: int
    compute_delay_sec: float
    parsed_candidate: Optional[str]


@dataclass
class EpisodeResult:
    idx: int
    gt: str
    pred: Optional[str]
    correct: bool
    policy: str
    visited_path: List[str]
    agent_records: List[AgentRecord]
    hop_records: List[HopRecord]
    total_compute_delay_sec: float
    total_communication_delay_sec: float
    total_orchestrator_compute_sec: float
    total_delay_sec: float
    total_compute_delay_sec_exclude_A_E: float
    total_delay_sec_exclude_A_E_compute: float
    orchestrator_outputs: List[Optional[str]]
    orchestrator_chosen_next: List[str]
    mean_quality_edge_map: Dict[str, float]
    pairwise_distances: Dict[str, float]


def append_hop(
    hop_records: List[HopRecord],
    src: str,
    dst: str,
    payload_bits: int,
    state_sample: LinkStateSample,
    state_table: Dict[str, LinkStateParams],
    rng: random.Random,
) -> float:
    breakdown = compute_comm_delay_from_state(
        payload_bits=payload_bits,
        state=state_sample.state,
        table=state_table,
        hidden_quality_score=state_sample.hidden_quality_score,
        rng=rng,
    )
    hop_records.append(
        HopRecord(
            src=src,
            dst=dst,
            link_state=breakdown.state,
            hidden_quality_score=breakdown.hidden_quality_score,
            throughput_bps=breakdown.throughput_bps,
            extra_delay_mean_sec=breakdown.extra_delay_mean_sec,
            extra_delay_std_sec=breakdown.extra_delay_std_sec,
            random_extra_delay_sec=breakdown.random_extra_delay_sec,
            message_bits=payload_bits,
            serialization_delay_sec=breakdown.serialization_delay_sec,
            communication_delay_sec=breakdown.total_delay_sec,
        )
    )
    return breakdown.total_delay_sec


def run_episode(
    model: SharedLLM,
    orchestrator_model: SharedLLM,
    example: Dict,
    idx: int,
    policy_name: str,
    max_new_tokens: int,
    estimator: DelayEstimator,
    example_seed: int,
    link_quality_cfg: LinkQualityConfig,
    state_table: Dict[str, LinkStateParams],
    orch_max_new_tokens: int = 256,
    orch_myopic_projection: bool = True,
    max_new_tokens_e: Optional[int] = None,
    agent_enable_thinking: bool = False,
    agent_e_enable_thinking: bool = False,
    orchestrator_enable_thinking: bool = False,
) -> EpisodeResult:
    e_max_new = max_new_tokens_e if max_new_tokens_e is not None else max_new_tokens

    dist_sym, edge_link_samples = precompute_edge_link_samples(example_seed)
    pairwise_distances = {f"{a}-{b}": dist_sym[(a, b)] for a, b in _UNDIRECTED_ROLE_PAIRS}
    orch = Orchestrator(
        policy_name=policy_name,
        estimator=estimator,
        state_table=state_table,
        myopic_projection=orch_myopic_projection,
    )

    visited = ["ENC"]
    remaining_intermediates = ["SYN", "ALT", "VER"]
    current_role = "ENC"

    agent_records: List[AgentRecord] = []
    hop_records: List[HopRecord] = []
    orchestrator_outputs: List[Optional[str]] = []
    orchestrator_chosen_next: List[str] = []

    total_compute = 0.0
    total_comm = 0.0
    total_orch_compute = 0.0

    prompt = build_agent_prompt("ENC", example, None, None)
    text, gen_tokens, compute_sec = model.generate(
        prompt, max_new_tokens, enable_thinking=agent_enable_thinking
    )
    message_bits = payload_bits_utf8(text)
    total_compute += compute_sec
    estimator.observe("ENC", compute_sec, gen_tokens, message_bits)
    agent_records.append(
        AgentRecord(
            role="ENC",
            role_name=ROLE_META["ENC"]["name"],
            prompt_preview=prompt[:250],
            generated_text=text,
            generated_tokens=gen_tokens,
            generated_utf8_bits=message_bits,
            compute_delay_sec=compute_sec,
            parsed_candidate=extract_answer(text),
        )
    )
    current_output = text
    current_message_bits = message_bits

    while remaining_intermediates:
        candidate_samples = {cand: edge_link_samples[(current_role, cand)] for cand in remaining_intermediates}
        candidate_states = {cand: samp.state for cand, samp in candidate_samples.items()}
        elapsed = total_compute + total_comm

        if len(remaining_intermediates) == 1:
            next_role = remaining_intermediates[0]
            orch_step_sec = 0.0
            orch_llm_raw = None
        else:
            decision, orch_step_sec, orch_llm_raw = orch.decide(
                current_role=current_role,
                current_message_bits=current_message_bits,
                visited=visited,
                remaining=list(remaining_intermediates),
                candidate_states=candidate_states,
                elapsed_without_orchestrator=elapsed,
                model=orchestrator_model,
                orch_max_new_tokens=orch_max_new_tokens,
                previous_agent_output=current_output,
                dist_sym=dist_sym,
                pairwise_distances=pairwise_distances,
                orchestrator_enable_thinking=orchestrator_enable_thinking,
            )
            next_role = decision.chosen_next

        total_orch_compute += orch_step_sec
        orchestrator_outputs.append(orch_llm_raw)
        orchestrator_chosen_next.append(next_role)

        hop_rng = hop_comm_random(example_seed, current_role, next_role)
        hop_delay = append_hop(
            hop_records=hop_records,
            src=current_role,
            dst=next_role,
            payload_bits=current_message_bits,
            state_sample=edge_link_samples[(current_role, next_role)],
            state_table=state_table,
            rng=hop_rng,
        )
        total_comm += hop_delay

        prompt = build_agent_prompt(next_role, example, current_output, current_role)
        text, gen_tokens, compute_sec = model.generate(
            prompt, max_new_tokens, enable_thinking=agent_enable_thinking
        )
        message_bits = payload_bits_utf8(text)
        total_compute += compute_sec
        estimator.observe(next_role, compute_sec, gen_tokens, message_bits)

        agent_records.append(
            AgentRecord(
                role=next_role,
                role_name=ROLE_META[next_role]["name"],
                prompt_preview=prompt[:250],
                generated_text=text,
                generated_tokens=gen_tokens,
                generated_utf8_bits=message_bits,
                compute_delay_sec=compute_sec,
                parsed_candidate=extract_answer(text),
            )
        )

        visited.append(next_role)
        remaining_intermediates.remove(next_role)
        current_role = next_role
        current_output = text
        current_message_bits = message_bits

    # last hop to FIN
    hop_rng = hop_comm_random(example_seed, current_role, "FIN")
    hop_delay = append_hop(
        hop_records=hop_records,
        src=current_role,
        dst="FIN",
        payload_bits=current_message_bits,
        state_sample=edge_link_samples[(current_role, "FIN")],
        state_table=state_table,
        rng=hop_rng,
    )
    total_comm += hop_delay

    prompt = build_agent_prompt("FIN", example, current_output, current_role)
    final_text, final_tokens, final_compute_sec = model.generate(
        prompt, e_max_new, enable_thinking=agent_e_enable_thinking
    )
    final_bits = payload_bits_utf8(final_text)
    total_compute += final_compute_sec
    estimator.observe("FIN", final_compute_sec, final_tokens, final_bits)

    agent_records.append(
        AgentRecord(
            role="FIN",
            role_name=ROLE_META["FIN"]["name"],
            prompt_preview=prompt[:250],
            generated_text=final_text,
            generated_tokens=final_tokens,
            generated_utf8_bits=final_bits,
            compute_delay_sec=final_compute_sec,
            parsed_candidate=extract_answer(final_text),
        )
    )
    visited.append("FIN")

    pred = extract_answer(final_text)
    gt = example["answer"]
    correct = pred == gt

    compute_exclude_ae = sum(
        rec.compute_delay_sec for rec in agent_records if rec.role not in ("ENC", "FIN")
    )

    mean_quality_edge_map = {
        f"{src}->{dst}": edge_link_samples[(src, dst)].hidden_quality_score
        for src in ALL_ROLES for dst in ALL_ROLES if src != dst
    }

    return EpisodeResult(
        idx=idx,
        gt=gt,
        pred=pred,
        correct=correct,
        policy=policy_name,
        visited_path=visited,
        agent_records=agent_records,
        hop_records=hop_records,
        total_compute_delay_sec=total_compute,
        total_communication_delay_sec=total_comm,
        total_orchestrator_compute_sec=total_orch_compute,
        total_delay_sec=total_compute + total_comm + total_orch_compute,
        total_compute_delay_sec_exclude_A_E=compute_exclude_ae,
        total_delay_sec_exclude_A_E_compute=compute_exclude_ae + total_comm + total_orch_compute,
        orchestrator_outputs=orchestrator_outputs,
        orchestrator_chosen_next=orchestrator_chosen_next,
        mean_quality_edge_map=mean_quality_edge_map,
        pairwise_distances=pairwise_distances,
    )


def summarize_results(results: List[EpisodeResult]) -> Dict[str, Any]:
    acc = sum(1 for r in results if r.correct) / max(len(results), 1)
    avg_total = statistics.mean(r.total_delay_sec for r in results) if results else 0.0
    avg_compute = statistics.mean(r.total_compute_delay_sec for r in results) if results else 0.0
    avg_comm = statistics.mean(r.total_communication_delay_sec for r in results) if results else 0.0
    avg_orch = statistics.mean(r.total_orchestrator_compute_sec for r in results) if results else 0.0
    avg_total_excl_ae = statistics.mean(r.total_delay_sec_exclude_A_E_compute for r in results) if results else 0.0

    path_counter = defaultdict(int)
    for r in results:
        path_counter["->".join(r.visited_path)] += 1

    return {
        "n": len(results),
        "accuracy": acc,
        "avg_total_delay_sec": avg_total,
        "avg_compute_delay_sec": avg_compute,
        "avg_comm_delay_sec": avg_comm,
        "avg_orchestrator_compute_sec": avg_orch,
        "avg_total_delay_sec_exclude_A_E_compute": avg_total_excl_ae,
        "path_counts": dict(path_counter),
    }


print("Cell 6 OK: pipeline execution and results loaded.")

In [ ]:
# =========================
# Main experiment loop
# =========================
# (argparse defaults converted to variables)

# --- Task / dataset ---
TASK = "strategyqa"
STRATEGYQA_DATASET_ID = "ChilleD/StrategyQA"  # <-- set to your HF dataset repo
STRATEGYQA_DATASET_CONFIG = None
SPLIT = "validation"
NUM_EXAMPLES = 30
SEED = 42

# --- Model ---
MODEL_NAME = "Qwen/Qwen3-4B"
DTYPE = "float16"
LOAD_IN_4BIT = False
QUANT_COMPUTE_DTYPE = "float16"
DEVICE_MAP = "auto"
TORCH_COMPILE = False
ATTN_IMPLEMENTATION = "auto"

# --- Generation ---
MAX_NEW_TOKENS = 96
MAX_NEW_TOKENS_E = 32
ORCH_MAX_NEW_TOKENS = 128

# --- Policy ---
POLICY = "role_only"  # choices: "role_only", "link_quality_aware", "link_quality_llm"

# --- Thinking ---
AGENT_ENABLE_THINKING = False
AGENT_E_ENABLE_THINKING = False
ORCHESTRATOR_ENABLE_THINKING = False

# --- Delay estimation ---
INITIAL_COMPUTE_SEC = 1.0
DEFAULT_MESSAGE_BYTES = 256

# --- Link quality config ---
EXCELLENT_THROUGHPUT_BPS = 4_000_000.0
GOOD_THROUGHPUT_BPS = 2_000_000.0
FAIR_THROUGHPUT_BPS = 500_000.0
POOR_THROUGHPUT_BPS = 100_000.0
EXCELLENT_EXTRA_DELAY_MEAN_MS = 2.0
GOOD_EXTRA_DELAY_MEAN_MS = 5.0
FAIR_EXTRA_DELAY_MEAN_MS = 20.0
POOR_EXTRA_DELAY_MEAN_MS = 60.0
EXTRA_DELAY_STD_MS = 2.0

# --- Run ---
print(f"Setting seed: {SEED}")
set_seed(SEED)

dataset_name = normalize_task_arg(TASK)
print(f"Loading dataset: {STRATEGYQA_DATASET_ID} (split={SPLIT})...")
raw_ds = load_examples(
    dataset_name=dataset_name,
    split=SPLIT,
    strategyqa_dataset_id=STRATEGYQA_DATASET_ID,
    strategyqa_dataset_config=STRATEGYQA_DATASET_CONFIG,
)

examples = [prepare_example(raw_ds[i], dataset_name) for i in range(min(NUM_EXAMPLES, len(raw_ds)))]
print(f"Prepared {len(examples)} examples.")

print(f"Loading model: {MODEL_NAME}...")
model = SharedLLM(
    model_name=MODEL_NAME,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    quant_compute_dtype=QUANT_COMPUTE_DTYPE,
    device_map=DEVICE_MAP,
    torch_compile=TORCH_COMPILE,
    attn_implementation=ATTN_IMPLEMENTATION,
)
orchestrator_model = model
print("Model loaded.")

cfg = LinkQualityConfig(
    excellent_throughput_bps=EXCELLENT_THROUGHPUT_BPS,
    good_throughput_bps=GOOD_THROUGHPUT_BPS,
    fair_throughput_bps=FAIR_THROUGHPUT_BPS,
    poor_throughput_bps=POOR_THROUGHPUT_BPS,
    excellent_extra_delay_mean_ms=EXCELLENT_EXTRA_DELAY_MEAN_MS,
    good_extra_delay_mean_ms=GOOD_EXTRA_DELAY_MEAN_MS,
    fair_extra_delay_mean_ms=FAIR_EXTRA_DELAY_MEAN_MS,
    poor_extra_delay_mean_ms=POOR_EXTRA_DELAY_MEAN_MS,
    extra_delay_std_ms=EXTRA_DELAY_STD_MS,
)
state_table = build_link_state_table(cfg)

estimator = DelayEstimator(
    roles=ALL_ROLES,
    initial_compute_sec=INITIAL_COMPUTE_SEC,
    max_new_tokens=MAX_NEW_TOKENS,
    default_message_bytes=DEFAULT_MESSAGE_BYTES,
    link_state_table=state_table,
)

results: List[EpisodeResult] = []
print(f"\nRunning {len(examples)} episodes with policy='{POLICY}'...\n")

for idx, ex in enumerate(examples):
    ep = run_episode(
        model=model,
        orchestrator_model=orchestrator_model,
        example=ex,
        idx=idx,
        policy_name=POLICY,
        max_new_tokens=MAX_NEW_TOKENS,
        estimator=estimator,
        example_seed=SEED + idx,
        link_quality_cfg=cfg,
        state_table=state_table,
        orch_max_new_tokens=ORCH_MAX_NEW_TOKENS,
        orch_myopic_projection=True,
        max_new_tokens_e=MAX_NEW_TOKENS_E,
        agent_enable_thinking=AGENT_ENABLE_THINKING,
        agent_e_enable_thinking=AGENT_E_ENABLE_THINKING,
        orchestrator_enable_thinking=ORCHESTRATOR_ENABLE_THINKING,
    )
    results.append(ep)

    print(f"[{idx:03d}] gt={ep.gt} pred={ep.pred} correct={ep.correct} path={'->'.join(ep.visited_path)} "
          f"total={ep.total_delay_sec:.3f}s compute={ep.total_compute_delay_sec:.3f}s "
          f"comm={ep.total_communication_delay_sec:.3f}s orch={ep.total_orchestrator_compute_sec:.3f}s")

summary = summarize_results(results)
print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))